# RAG from Scratch: Embeddings, Vector Search, and a Working Pipeline

Companion notebook for **Chapter 44** of *Large Language Models from the Ground Up*.

You will build a complete retrieval-augmented generation (RAG) pipeline in plain Python + NumPy:

1. a **toy embedder** (hash-based bag of words) that turns any text into one vector;
2. **cosine similarity**, verified by hand on a 3-D example;
3. brute-force **vector search** over a five-document corpus;
4. **prompt assembly** — the grounded prompt you'd hand to any chat model.

The toy embedder exists so you can see every moving part. **A real system swaps that one function for a learned embedding model** — the final section shows the exact upgrade (`sentence-transformers`), to run here on Colab.


## The corpus

Five tiny documents about a fictional company, Cascara Coffee Roasters. Each is short enough to be its own chunk — Chapter 44 covers chunking (sizes, overlap) for real documents.

In [1]:
import re, hashlib
import numpy as np

DOCS = {
 "returns": "Cascara Coffee accepts returns within 30 "
   "days of delivery. Unopened bags get a full refund; "
   "opened bags get store credit.",
 "shipping": "Orders ship from our Portland roastery "
   "every Monday and Thursday. Standard shipping takes "
   "3 to 5 business days and is free over $35.",
 "history": "Cascara Coffee Roasters was founded in "
   "2019 by Maya Okonkwo, a former chemist who began "
   "roasting beans in her garage.",
 "subscription": "The subscription plan delivers a "
   "fresh 12-ounce bag every two weeks for $18. "
   "Subscribers can pause or cancel at any time.",
 "beans": "Our signature blend, Fog Cutter, combines "
   "beans from Ethiopia and Colombia, roasted to a "
   "medium profile with notes of cherry and cocoa.",
}
print(len(DOCS), "documents")

5 documents


## The toy embedder

One vector per text: hash each meaningful word to one of 4,096 slots, count, then scale the vector to length 1 (so dot products *are* cosine similarities).

This is word-overlap search dressed as vector search — deliberately. Every real pipeline has exactly this shape with `embed` replaced by a learned transformer trained **contrastively** (similar meaning → nearby vectors).

In [2]:
DIM = 4096         # slots per vector
STOP = set("a an and at by do for from get i in is "
           "it my of or our the to was with".split())

def words(text):
    ws = re.findall(r"[a-z0-9]+", text.lower())
    return [w.rstrip("s") for w in ws if w not in STOP]

def embed(text):
    """Toy embedder: hash each word to one of DIM
    slots, count, scale to length 1. A real system
    replaces THIS FUNCTION with a learned model."""
    v = np.zeros(DIM)
    for w in words(text):
        d = hashlib.sha256(w.encode()).hexdigest()
        v[int(d, 16) % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

s = "Unopened bags get a full refund"
v = embed(s)
print("kept words :", words(s))
print("vector dims:", v.shape[0])
print("length     :", np.linalg.norm(v))

kept words : ['unopened', 'bag', 'full', 'refund']
vector dims: 4096
length     : 1.0


## Cosine similarity, verified

Chapter 44 works this 3-D example by hand: all three vectors have length 3, so `cos(a, b) = (a · b) / 9`. Check the arithmetic.

In [3]:
a = np.array([2., 1., 2.])
b = np.array([1., 2., 2.])
c = np.array([2., -2., 1.])

def cosine(x, y):
    return x @ y / (np.linalg.norm(x)
                    * np.linalg.norm(y))

print("cos(a, b) =", cosine(a, b))  # by hand: 8/9
print("cos(a, c) =", cosine(a, c))  # by hand: 4/9

cos(a, b) = 0.8888888888888888
cos(a, c) = 0.4444444444444444


## Index and search

Indexing = embed every document once, stack into a matrix. Search = one matrix–vector multiply (`index @ q`) — **brute force**, which is exact and fast up to about a million vectors.

In [4]:
names = list(DOCS)
index = np.stack([embed(DOCS[n]) for n in names])
print("index shape:", index.shape)

def retrieve(question, k=2):
    scores = index @ embed(question)  # cosine scores
    best = np.argsort(-scores)[:k]
    return [(scores[i], names[i]) for i in best]

q = "How many days do I have to return a bag of coffee?"
for score, name in retrieve(q, k=5):
    print(f"  {score:.3f}  {name}")

index shape: (5, 4096)
  0.445  returns
  0.098  history
  0.092  shipping
  0.092  subscription
  0.000  beans


Every score has a reason: *returns* shares the load-bearing words (days, return, bag, coffee); *history* and *subscription* brush the question with one word each; *beans* shares nothing and scores exactly 0.

## Assemble the grounded prompt

Top-k chunks, source tags for citation, and a licensed "I don't know" — then hand the string to **any chat model**. That final generation step is the only part this notebook outsources.

In [5]:
def build_prompt(question, k=2):
    hits = retrieve(question, k)
    ctx = "\n".join(f"[{n}] {DOCS[n]}"
                    for _, n in hits)
    return ("Answer using ONLY the context below. "
            "If the answer is not there, say you "
            f"do not know.\n\nContext:\n{ctx}\n\n"
            f"Question: {question}\nAnswer:")

print(build_prompt(q))

Answer using ONLY the context below. If the answer is not there, say you do not know.

Context:
[returns] Cascara Coffee accepts returns within 30 days of delivery. Unopened bags get a full refund; opened bags get store credit.
[history] Cascara Coffee Roasters was founded in 2019 by Maya Okonkwo, a former chemist who began roasting beans in her garage.

Question: How many days do I have to return a bag of coffee?
Answer:


Paste that prompt into ChatGPT, Claude, or Gemini and you get a grounded answer along the lines of:

> You have 30 days from delivery to return a bag. Unopened bags receive a full refund; opened bags receive store credit. \[returns\]

## The instructive failure

Ask the same fact two ways. The document says *founded*, not *started*; *Cascara Coffee Roasters*, not *the company*. The toy embedder matches **words**, not meaning — this is exactly the gap learned embedding models close.

In [6]:
for q2 in ["Who started the company?",
           "Who founded Cascara Coffee Roasters?"]:
    print(q2)
    for score, name in retrieve(q2, k=2):
        print(f"  {score:.3f}  {name}")

Who started the company?
  0.149  history
  0.000  returns
Who founded Cascara Coffee Roasters?
  0.577  history
  0.211  returns


## Your turn

Edit the question below. Before running: predict which document wins, and whether the score will be strong (shared rare words) or weak (paraphrase).

In [7]:
my_q = "Can I pause my subscription?"
for score, name in retrieve(my_q, k=5):
    print(f"  {score:.3f}  {name}")

  0.420  subscription
  0.000  returns
  0.000  shipping
  0.000  history
  0.000  beans


## The real thing: swap in a learned embedder

Everything below upgrades `embed` to a real embedding model — `all-MiniLM-L6-v2`, a small transformer (~22M parameters, 384 dimensions) trained contrastively on over a billion text pairs. **Run these cells on Colab** (first run downloads the model). Retrieval, prompt assembly, and everything downstream stay unchanged — the pipeline you built is the real pipeline.

In [ ]:
# Colab: install once (takes about a minute)
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

st_index = model.encode(list(DOCS.values()),
                        normalize_embeddings=True)
print("index shape:", st_index.shape)

def st_retrieve(question, k=2):
    qv = model.encode([question],
                      normalize_embeddings=True)[0]
    scores = st_index @ qv   # cosine, as before
    best = np.argsort(-scores)[:k]
    return [(scores[i], names[i]) for i in best]

In [ ]:
# The paraphrase the toy embedder fumbled:
for score, name in st_retrieve(
        "Who started the company?", k=5):
    print(f"  {score:.3f}  {name}")
# Expect: history wins clearly, despite sharing
# no content words with the question.

## Where to go next

- Replace `DOCS` with 10–20 chunks of your own notes and re-run both embedders (Chapter 44, exercise 5).
- Add overlap-aware chunking for longer documents, then a reranker, then hybrid BM25 + vector search — Chapter 44 explains each upgrade.
- All book links: `books.wazeem.com/llm`.